In [12]:
import pandas as pd
import numpy as np

df=pd.read_csv('Ass.csv')
df
dff=pd.read_csv('Ass2.csv')

In [13]:
df = df.drop(columns=['Temp', 'Humidity(%)','Day'], axis=1)
df
dff = dff.drop(columns=['Temp', 'Humidity(%)','Day'], axis=1)
dff

,Outlook,Wind,PlayTennis,Temp_Category,Humidity_Category
0,Sunny,Strong,Yes,Hot,High
1,Sunny,Strong,Yes,Hot,High
2,Overcast,Weak,No,Hot,High
3,Overcast,Strong,No,Mild,High
4,Rain,Strong,No,Cool,Normal
5,Rain,Weak,No,Cool,Normal
6,Sunny,Strong,Yes,Mild,High
7,Overcast,Weak,Yes,Cool,Normal


In [14]:
from sklearn.preprocessing import LabelEncoder

# List of columns to encode
columns_to_encode = ['Outlook', 'Wind', 'PlayTennis', 'Temp_Category', 'Humidity_Category']

# Create a new LabelEncoder instance and apply it to each column
for column in columns_to_encode:
    le = LabelEncoder()  # New instance for each column
    df[column] = le.fit_transform(df[column])

# Display the resulting DataFrame

for column in columns_to_encode:
    le = LabelEncoder()  # New instance for each column
    dff[column] = le.fit_transform(dff[column])
    
df



,Outlook,Wind,PlayTennis,Temp_Category,Humidity_Category
0,2,1,0,1,0
1,2,0,0,1,0
2,0,1,1,1,0
3,1,1,1,0,0
4,1,1,1,0,1
5,1,0,0,0,1
6,0,0,1,0,1
7,2,1,0,2,0
8,2,1,1,2,1
9,1,1,1,0,1


In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn import tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from math import log2

def calculate_entropy(data):
    """Calculate entropy of a dataset"""
    if len(data) == 0:
        return 0

    # Count occurrences of each class
    class_counts = data.value_counts()
    total_samples = len(data)

    entropy = 0
    for count in class_counts:
        probability = count / total_samples
        if probability > 0:
            entropy -= probability * log2(probability)

    return entropy

def main():
    # Load and preprocess data (same as in your notebook)
    df = pd.read_csv('Ass.csv')
    dff = pd.read_csv('Ass2.csv')

    df = df.drop(columns=['Temp', 'Humidity(%)','Day'], axis=1)
    dff = dff.drop(columns=['Temp', 'Humidity(%)','Day'], axis=1)

    # List of columns to encode
    columns_to_encode = ['Outlook', 'Wind', 'PlayTennis', 'Temp_Category', 'Humidity_Category']

    # Create a new LabelEncoder instance and apply it to each column
    for column in columns_to_encode:
        le = LabelEncoder()  # New instance for each column
        df[column] = le.fit_transform(df[column])

    for column in columns_to_encode:
        le = LabelEncoder()  # New instance for each column
        dff[column] = le.fit_transform(dff[column])

    # Prepare features and target
    X = df.drop(columns=['PlayTennis'], axis=1)
    Y = df['PlayTennis']
    x = dff.drop(columns=['PlayTennis'], axis=1)
    y = dff['PlayTennis']

    # Create and train the model
    model = tree.DecisionTreeClassifier()
    model.fit(X, Y)

    # Model evaluation
    y_pred = model.predict(x)
    print("=== MODEL PERFORMANCE METRICS ===")
    print(f"Accuracy: {accuracy_score(y, y_pred):.4f}")
    print(f"Precision: {precision_score(y, y_pred):.4f}")
    print(f"Recall: {recall_score(y, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y, y_pred):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y, y_pred)}")

    # === ENTROPY AND INFORMATION GAIN ANALYSIS ===
    print("\n" + "="*50)
    print("ATTRIBUTE ENTROPY AND INFORMATION GAIN ANALYSIS")
    print("="*50)

    # Calculate entropy for each attribute
    print("\n=== ATTRIBUTE ENTROPY ANALYSIS ===")
    print(f"Target variable (PlayTennis) entropy: {calculate_entropy(Y):.4f}")

    feature_names = X.columns.tolist()
    print("\nIndividual Attribute Entropies:")
    for feature in feature_names:
        entropy = calculate_entropy(X[feature])
        print(f"  {feature}: {entropy:.4f}")

    # Calculate information gain for each attribute
    print("\n=== INFORMATION GAIN ANALYSIS ===")
    target_entropy = calculate_entropy(Y)

    print(f"Target entropy: {target_entropy:.4f}")
    print("\nInformation Gain for each attribute:")

    for feature in feature_names:
        # Calculate weighted entropy for each feature value
        weighted_entropy = 0
        for value in X[feature].unique():
            subset = Y[X[feature] == value]
            weight = len(subset) / len(Y)
            weighted_entropy += weight * calculate_entropy(subset)

        information_gain = target_entropy - weighted_entropy
        print(f"  {feature}: {information_gain:.4f}")

    # Show which attribute has the highest information gain
    print("\n=== ATTRIBUTE SELECTION BASED ON INFORMATION GAIN ===")
    gains = {}
    for feature in feature_names:
        weighted_entropy = 0
        for value in X[feature].unique():
            subset = Y[X[feature] == value]
            weight = len(subset) / len(Y)
            weighted_entropy += weight * calculate_entropy(subset)
        gains[feature] = target_entropy - weighted_entropy

    best_attribute = max(gains, key=gains.get)
    print(f"Best attribute to split on (highest information gain): {best_attribute}")
    print(f"Information gain: {gains[best_attribute]:.4f}")

    # Display attribute ranking
    print("\nAttribute ranking by information gain:")
    sorted_gains = sorted(gains.items(), key=lambda x: x[1], reverse=True)
    for i, (attr, gain) in enumerate(sorted_gains, 1):
        print(f"  {i}. {attr}: {gain:.4f}")

if __name__ == "__main__":
    main()


=== MODEL PERFORMANCE METRICS ===
Accuracy: 0.2500
Precision: 0.2500
Recall: 0.2500
F1 Score: 0.2500
Confusion Matrix:
[[1 3]
 [3 1]]

ATTRIBUTE ENTROPY AND INFORMATION GAIN ANALYSIS

=== ATTRIBUTE ENTROPY ANALYSIS ===
Target variable (PlayTennis) entropy: 0.9403

Individual Attribute Entropies:
  Outlook: 1.5774
  Wind: 0.9852
  Temp_Category: 1.5306
  Humidity_Category: 1.0000

=== INFORMATION GAIN ANALYSIS ===
Target entropy: 0.9403

Information Gain for each attribute:
  Outlook: 0.2467
  Wind: 0.0481
  Temp_Category: 0.1182
  Humidity_Category: 0.1518

=== ATTRIBUTE SELECTION BASED ON INFORMATION GAIN ===
Best attribute to split on (highest information gain): Outlook
Information gain: 0.2467

Attribute ranking by information gain:
  1. Outlook: 0.2467
  2. Humidity_Category: 0.1518
  3. Temp_Category: 0.1182
  4. Wind: 0.0481


In [17]:
import pandas as pd
import numpy as np

def main():
    # Load the dataset
    df = pd.read_csv('Ass.csv')

    print("TENNIS DATASET - DUMMY TABLES ANALYSIS")
    print("="*50)
    print("Target: PlayTennis (Yes/No)")
    print("="*50)

    # Get attributes to analyze
    attributes = ['Outlook', 'Temp_Category', 'Humidity_Category', 'Wind']

    for attribute in attributes:
        print(f"\n{attribute.upper()} - DUMMY TABLE")
        print("-" * 30)

        # Create contingency table
        dummy_table = pd.crosstab(df[attribute], df['PlayTennis'])
        print(dummy_table)

        # Add totals
        dummy_table_with_totals = pd.crosstab(df[attribute], df['PlayTennis'], margins=True, margins_name="Total")
        print(f"\n{attribute} with totals:")
        print(dummy_table_with_totals)

        # Calculate percentages
        print(f"\n{attribute} - Percentage breakdown:")
        print("-" * 30)
        for value in df[attribute].unique():
            yes_count = len(df[(df[attribute] == value) & (df['PlayTennis'] == 'Yes')])
            no_count = len(df[(df[attribute] == value) & (df['PlayTennis'] == 'No')])
            total = yes_count + no_count

            if total > 0:
                yes_pct = (yes_count / total) * 100
                no_pct = (no_count / total) * 100
                print(f"  {value}:")
                print(f"    Yes: {yes_count}/{total} ({yes_pct:.1f}%)")
                print(f"    No:  {no_count}/{total} ({no_pct:.1f}%)")

if __name__ == "__main__":
    main()


TENNIS DATASET - DUMMY TABLES ANALYSIS
Target: PlayTennis (Yes/No)

OUTLOOK - DUMMY TABLE
------------------------------
PlayTennis  No  Yes
Outlook            
Overcast     0    4
Rain         2    3
Sunny        3    2

Outlook with totals:
PlayTennis  No  Yes  Total
Outlook                   
Overcast     0    4      4
Rain         2    3      5
Sunny        3    2      5
Total        5    9     14

Outlook - Percentage breakdown:
------------------------------
  Sunny:
    Yes: 2/5 (40.0%)
    No:  3/5 (60.0%)
  Overcast:
    Yes: 4/4 (100.0%)
    No:  0/4 (0.0%)
  Rain:
    Yes: 3/5 (60.0%)
    No:  2/5 (40.0%)

TEMP_CATEGORY - DUMMY TABLE
------------------------------
PlayTennis     No  Yes
Temp_Category         
Cool            1    5
Hot             2    1
Mild            2    3

Temp_Category with totals:
PlayTennis     No  Yes  Total
Temp_Category                
Cool            1    5      6
Hot             2    1      3
Mild            2    3      5
Total           5    9 

In [18]:
from sklearn import tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix

X=df.drop(columns=['PlayTennis'],axis=1)
Y=df['PlayTennis']
x=dff.drop(columns=['PlayTennis'],axis=1)
y=dff['PlayTennis']

model=tree.DecisionTreeClassifier()
model.fit(X,Y)

model.score(x,y)

y_pred=model.predict(x)
print(accuracy_score(y,y_pred))
print(precision_score(y,y_pred))
print(recall_score(y,y_pred))
print(f1_score(y,y_pred))
print(confusion_matrix(y,y_pred))


0.25
0.25
0.25
0.25
[[1 3]
 [3 1]]
